## Who worked on this part?

Mohammed AbuQamar

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from pycaret.regression import setup, compare_models, pull

# 1. Load the Time-Series Data (Created in Notebook 2)
train = pd.read_csv('citibike_train_dense.csv')
test = pd.read_csv('citibike_test_dense.csv')

# Define features and target (Predicting raw ride volume based on time)
features = ['hour', 'day_of_week', 'is_weekend']
target = 'ride_count'

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

print(f"Training shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Testing shapes: X={X_test.shape}, y={y_test.shape}\n")

# --- 2. The Quick Baseline Model (Linear Regression) ---
# As requested by the project prompt to unblock the deployment team
print("--- Training Baseline: Linear Regression ---")
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))
print(f"Baseline Linear Regression RMSE: {baseline_rmse:.2f} rides per hour\n")


# --- 3. Automated ML Comparison (PyCaret) ---
# Required by the MVP rubric
print("--- Running PyCaret AutoML ---")
# PyCaret expects the target in the same dataframe
pycaret_data = train[features + [target]]

# Initialize PyCaret setup (silent=True to prevent it asking for user input in notebooks)
reg_setup = setup(data=pycaret_data, target=target, session_id=42, verbose=False)

# Compare models to find the top performers based on RMSE
best_pycaret_model = compare_models(sort='RMSE', n_select=1)
pycaret_results = pull()
print("Top 3 PyCaret Models:")
print(pycaret_results[['Model', 'RMSE', 'MAE', 'R2']].head(3))
print("\n")


# --- 4. Tuned Ensemble Model (XGBoost Regressor) ---
# XGBoost is explicitly recommended in your slides (Chapter 6) for reducing bias
print("--- Training Tuned XGBoost Regressor ---")
xgb_model = XGBRegressor(
    n_estimators=200, 
    max_depth=6, 
    learning_rate=0.05, 
    random_state=42,
    objective='reg:squarederror'
)

xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))

print(f"Tuned XGBoost RMSE: {xgb_rmse:.2f} rides per hour\n")


# --- 5. Visualizing the Forecast vs Actual ---
# Plot a 48-hour window from the test set to see how the model tracks the daily spikes
window_size = 48
plt.figure(figsize=(14, 5))
plt.plot(y_test.values[:window_size], label='Actual Demand', color='black', linewidth=2)
plt.plot(xgb_preds[:window_size], label='XGBoost Forecast', color='teal', linestyle='--')
plt.plot(baseline_preds[:window_size], label='Baseline Forecast', color='orange', linestyle=':')

plt.title('Citi Bike Hourly Demand Forecast (48-Hour Window)', fontsize=16)
plt.xlabel('Hours into Test Set', fontsize=12)
plt.ylabel('Total Ride Count', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

# --- 6. Export Final Local Model ---
import joblib
joblib.dump(xgb_model, 'citibike_xgboost_model.pkl')
print("Best local model exported as 'citibike_xgboost_model.pkl'")

ModuleNotFoundError: No module named 'sklearn'